# Module 3：把图片接入 PyTorch

依据你提供的 Module 3 讲义设计的本地练习，不是官方实验文件。按 Data Access → Transform Pipelines → DataLoader → Bugproof Pipelines 的顺序进行。

这次重点是数据流程，不训练分类器，也不比较模型准确率。你需要补全四组 TODO；遇到 `NotImplementedError`，说明到了练习位置。填写后删掉对应的 `raise` 行。

## 材料与运行方式

- 在 VS Code 选择原来已安装 torch、torchvision、matplotlib 的 `.venv` 内核。
- 第一个代码单元会在当前工作目录创建 `module3_practice_data`，含 30 张简单的示意花朵图片、标签 CSV，以及 3 条故意有问题的样本。不下载数据，不安装软件。
- 图片是本地绘制的教学素材，不是 Oxford 102 原始照片；编号、大小差异、灰度图片和错误案例用于练习同样的数据管理方法。
- 官方标签文件是 `.mat`；这里用可直接阅读的 CSV 简化材料读取。CSV 读取和画图已提供，练习重点仍是 Dataset、transform、DataLoader 与错误检查。
- 重复运行会复用本练习创建的目录。如果同名目录已存在且不属于本练习，准备单元会停止，避免覆盖。
- 新建 notebook 有自己的内核状态；请从上向下运行。


In [ ]:
from pathlib import Path
import csv
import math
import torch
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torchvision import transforms
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

DATA_DIR = Path.cwd() / 'module3_practice_data'
marker = DATA_DIR / 'module3_demo_v1.txt'

def read_rows(path):
    with open(path, encoding='utf-8', newline='') as file:
        return list(csv.DictReader(file))

def write_rows(path, rows):
    with open(path, 'w', encoding='utf-8', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=['filename', 'label'])
        writer.writeheader()
        writer.writerows(rows)

def prepare_materials():
    if DATA_DIR.exists():
        if not marker.is_file():
            raise RuntimeError('同名目录已存在。请把 DATA_DIR 改为一个新的练习目录。')
        print('复用已有练习材料：', DATA_DIR)
        return

    (DATA_DIR / 'images').mkdir(parents=True)
    rows = []
    colors = [(225, 100, 100), (230, 195, 80), (155, 110, 220)]
    for i in range(30):
        width = 300 + (i % 5) * 37
        height = 220 + (i % 4) * 29
        image = Image.new('RGB', (width, height), (35, 65, 45))
        draw = ImageDraw.Draw(image)
        cx, cy = width // 2, height // 2 - 10
        radius = min(width, height) * 0.15
        draw.line((cx, cy, cx, height - 12), fill=(85, 150, 75), width=9)
        for petal in range(6):
            angle = petal * math.pi / 3 + (i % 3) * 0.1
            px = cx + radius * math.cos(angle)
            py = cy + radius * math.sin(angle)
            r = radius * 0.72
            draw.ellipse((px-r, py-r, px+r, py+r), fill=colors[i % 3])
        draw.ellipse((cx-15, cy-15, cx+15, cy+15), fill=(110, 75, 35))
        if i % 7 == 0:
            image = image.convert('L')  # 故意混入灰度图，Dataset 将统一成 RGB。
        filename = f'image_{i + 1:05d}.png'
        image.save(DATA_DIR / 'images' / filename)
        rows.append({'filename': filename, 'label': i % 3 + 1})

    write_rows(DATA_DIR / 'labels.csv', rows)
    # 三条故障：文件损坏、尺寸过小、文件不存在。
    (DATA_DIR / 'images' / 'broken.png').write_bytes(b'not an image')
    Image.new('RGB', (16, 20), 'white').save(DATA_DIR / 'images' / 'tiny.png')
    issues = [
        {'filename': 'broken.png', 'label': 1},
        {'filename': 'tiny.png', 'label': 2},
        {'filename': 'missing.png', 'label': 3},
    ]
    write_rows(DATA_DIR / 'labels_with_issues.csv', rows + issues)
    marker.write_text('Module 3 offline teaching fixture v1', encoding='utf-8')
    print('练习材料创建完成：', DATA_DIR)

prepare_materials()
rows = read_rows(DATA_DIR / 'labels.csv')
print('正常样本数量：', len(rows))
print('第一条记录：', rows[0])


## 先看懂材料中的新写法

一条 CSV 记录会读成字典，例如 `{'filename': 'image_00001.png', 'label': '1'}`。`row['filename']` 取文件名；`int(row['label'])` 将文本 `'1'` 转成整数 `1`。

`Path(root_dir)` 创建路径对象；路径对象之间的 `/` 表示拼接路径，如 `root / 'images' / filename`。这里不是数值除法。

`with Image.open(path) as original:` 表示打开一张图片，并在代码块结束时关闭文件；`original.convert('RGB')` 返回一张 RGB 图片。图片读取部分已给出。

标签 CSV 使用 1、2、3；模型类别编号使用 0、1、2，需要减 1。每条记录的文件名已经保存好；读取时按记录查找，不重新编号。这也适用于以后过滤坏样本、文件编号出现空缺的情况。


## 练习 A：自定义 Dataset

完成三个 TODO：

1. `__len__` 返回记录数量。
2. `__getitem__` 从指定记录取出文件名，拼出图片路径 `image_path`。
3. 从记录取标签，转成整数，再改为从 0 开始的类别编号，赋给 `label`。

`__init__` 保存路径、标签记录、处理步骤；取出一个样本时才打开对应图片，这就是按需加载。`dataset[0]` 会使用 `__getitem__(0)`，`len(dataset)` 会使用 `__len__()`。


In [ ]:
class MiniFlowersDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.img_dir = self.root_dir / 'images'
        self.records = read_rows(self.root_dir / 'labels.csv')
        self.transform = transform

    def __len__(self):
        # TODO A1：返回有多少条记录。
        raise NotImplementedError('A1：请补全 __len__')

    def __getitem__(self, idx):
        row = self.records[idx]
        # TODO A2：拼接目录与 row 中的文件名，结果赋给 image_path。
        raise NotImplementedError('A2：请补全 image_path')

        with Image.open(image_path) as original:
            image = original.convert('RGB')

        # TODO A3：从 row 取标签，转成整数并减 1，结果赋给 label。
        raise NotImplementedError('A3：请补全 label')

        if self.transform is not None:
            image = self.transform(image)
        return image, label

raw_dataset = MiniFlowersDataset(DATA_DIR)
image, label = raw_dataset[0]
print('样本数量：', len(raw_dataset))
print('第一张图片大小（宽、高）：', image.size)
print('第一张图片模式：', image.mode)
print('第一张标签：', label)
assert len(raw_dataset) == 30
assert image.mode == 'RGB'
assert label == 0
plt.imshow(image)
plt.axis('off')
plt.show()


## 练习 B：统一图片接口

创建 `eval_transform`，用 `transforms.Compose` 按讲义组合四步：

1. 短边缩放到 256，保持宽高比。
2. 从中心裁出 224×224。
3. 转成张量。
4. 用下面给定的三通道参数进行标准化。

可以使用的工具是 `Resize`、`CenterCrop`、`ToTensor`、`Normalize`。`Resize(256)` 与 `Resize((256, 256))` 的含义不同。

`ToTensor` 处理本练习的 8 位图片时，把像素缩放到 0～1，形状变成 `[C,H,W]`；`Normalize` 按通道执行 `(像素-mean)/std`，因此之后可以有负数。给定参数不保证这组练习图片的实际均值恰好为 0。


In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

# TODO B：创建 eval_transform，使用上述四步。
raise NotImplementedError('B：请创建 eval_transform')

eval_base = MiniFlowersDataset(DATA_DIR, transform=eval_transform)
image_tensor, label = eval_base[0]
print('处理后形状：', image_tensor.shape)
print('数值类型：', image_tensor.dtype)
print('最小值：', image_tensor.min().item())
print('最大值：', image_tensor.max().item())
assert tuple(image_tensor.shape) == (3, 224, 224)
assert image_tensor.dtype == torch.float32
assert torch.isfinite(image_tensor).all()


## 训练增强与数据划分：这一段已提供

训练图片加入轻微的水平翻转、旋转和亮度变化；验证、测试沿用固定处理。本练习是示意花朵，翻转不改变其类别；对于数字、文字等任务，不能直接照搬花朵的所有增强方式。

重要：`random_split` 得到的 Subset 共享原来的 Dataset。若对同一个底层对象修改 `transform`，其他子集也会受影响。这里创建两个 Dataset 对象，分别保存训练和评估处理，再让它们使用各自子集的索引。

下面的 `.indices` 是子集在原始数据中的编号列表；`Subset(数据集, 编号列表)` 表示按这份名单取样。数据划分的作用是分配样本；transform 的作用是处理样本，是两个不同环节。


In [ ]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2),
    eval_transform
])

# 30 张图按 70% / 15% / 剩余划分，得到 21 / 4 / 5。
train_size = int(0.7 * len(raw_dataset))
val_size = int(0.15 * len(raw_dataset))
test_size = len(raw_dataset) - train_size - val_size
train_part, val_part, test_part = random_split(
    raw_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_base = MiniFlowersDataset(DATA_DIR, transform=train_transform)
train_dataset = Subset(train_base, train_part.indices)
val_dataset = Subset(eval_base, val_part.indices)
test_dataset = Subset(eval_base, test_part.indices)
print('训练、验证、测试样本数：',
      len(train_dataset), len(val_dataset), len(test_dataset))
assert not (set(train_part.indices) & set(val_part.indices))
assert not (set(train_part.indices) & set(test_part.indices))
assert not (set(val_part.indices) & set(test_part.indices))


## 练习 C：创建 DataLoader，并检查完整一轮

创建三个 loader，名称为 `train_loader`、`val_loader`、`test_loader`，都设置 `batch_size=8, num_workers=0`。训练打乱顺序，验证和测试保持顺序。

`num_workers=0` 表示先在当前进程读取数据，方便理解调用和错误日志。这里读取、变换放在 CPU 完成即可，不需要把样本放到 GPU。

先预测一下：训练一轮有几批、最后一批有几张？再看下面的输出。`len(dataset)` 是样本数，`len(loader)` 是批次数。


In [ ]:
# TODO C：创建 train_loader、val_loader、test_loader。
raise NotImplementedError('C：请创建三个 DataLoader')

images, labels = next(iter(train_loader))
print('第一批图片：', images.shape, images.dtype)
print('第一批标签：', labels.shape, labels.dtype)
assert tuple(images.shape) == (8, 3, 224, 224)
assert tuple(labels.shape) == (8,)
assert labels.dtype == torch.int64
assert labels.min().item() >= 0 and labels.max().item() <= 2

batch_sizes = []
for images, labels in train_loader:
    batch_sizes.append(images.shape[0])
print('训练每批图片数量：', batch_sizes)
print('一轮共读取多少张：', sum(batch_sizes))
assert sum(batch_sizes) == len(train_dataset)
print('训练、验证、测试的批次数：',
      len(train_loader), len(val_loader), len(test_loader))


## 看看增强实际做了什么

画图代码已提供，不要求默写。反复取同一个训练样本时，原始图片和标签不变，随机处理结果可以不同。验证使用固定处理，重复取样的张量应相同。

`permute(1,2,0)` 把张量从 `[C,H,W]` 调整成画图使用的 `[H,W,C]`；`clamp(0,1)` 将显示值限制在 0～1。这里先撤销标准化再显示。


In [ ]:
mean_tensor = torch.tensor(MEAN).view(3, 1, 1)
std_tensor = torch.tensor(STD).view(3, 1, 1)
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax in axes:
    sample, label = train_dataset[0]
    displayed = (sample * std_tensor + mean_tensor).clamp(0, 1)
    ax.imshow(displayed.permute(1, 2, 0))
    ax.set_title('label=' + str(label))
    ax.axis('off')
plt.tight_layout()
plt.show()

first, first_label = val_dataset[0]
second, second_label = val_dataset[0]
assert torch.equal(first, second) and first_label == second_label
print('验证集重复读取结果一致。')


## 练习 D：扫描坏图片，记录错误

`labels_with_issues.csv` 在原来的 30 条正常记录后，加入了损坏文件、过小图片、缺失文件三条记录。

- `try`：尝试执行可能失败的操作。
- `raise ValueError(...)`：主动报告输入不符合要求。
- `except (OSError, ValueError) as error`：捕获文件/图片读取错误和检查失败，把错误对象命名为 `error`。
- `errors.append({...})`：把记录的文件名和错误原因加入列表。

请补上大小检查：只要宽或高小于 32，就抛出 `ValueError`。其余扫描代码已提供。

这里采用“先扫描、记录并筛除坏样本”的方法。讲义中递归改取下一张是演示处理思路，但实际会改变采样次数，全部样本损坏时也无法停止。记录并检查问题，比直接用零张量或任意图片顶替更容易追踪。


In [ ]:
def scan_images(root_dir, records):
    valid_rows = []
    errors = []
    for row in records:
        image_path = Path(root_dir) / 'images' / row['filename']
        try:
            with Image.open(image_path) as image:
                image.load()  # 真正读取像素，触发部分损坏文件的错误。
                width, height = image.size
                # TODO D：宽或高小于 32 时，用 raise ValueError 报告过小。
                raise NotImplementedError('D：请补上图片大小检查')
            valid_rows.append(row)
        except (OSError, ValueError) as error:
            errors.append({
                'filename': row['filename'],
                'reason': str(error)
            })
    return valid_rows, errors

all_rows = read_rows(DATA_DIR / 'labels_with_issues.csv')
valid_rows, errors = scan_images(DATA_DIR, all_rows)
print('扫描记录数：', len(all_rows))
print('正常记录数：', len(valid_rows))
print('错误记录数：', len(errors))
for error in errors:
    print(error['filename'], '->', error['reason'])
assert len(valid_rows) == 30 and len(errors) == 3


## 集中提交

完成 A～D 后，一次发来：

1. 你补的代码。
2. 单张图片与第一批图片的 shape / dtype。
3. 训练集每批图片数量，以及一轮样本总数。
4. 三条错误记录。

如果新语法挡住了你，可以先发不懂的行，先解释再继续。无需追求画图代码的熟练程度，也不用开始训练模型。

理解检查也集中回答：`dataset[7]` 使用哪个方法？为什么 `__init__` 只读一次标签表，而 `__getitem__` 按需读一张图片？为什么训练增强和验证处理要分开？

依据：用户提供的 Module 3 讲义。补充说明参考 [PyTorch 数据接口](https://docs.pytorch.org/docs/stable/data.html)、[Normalize](https://docs.pytorch.org/vision/stable/generated/torchvision.transforms.Normalize.html)。
